In [1]:
# Install pip packages
!mkdir ~/tmpdir
!TMPDIR=~/tmpdir python3 -m pip install --upgrade --no-cache-dir "sagemaker<3" pandas numpy matplotlib seaborn scikit-learn "torch==2.6" torchvision boto3 pandas

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 158.8 MB/s  0:00:09eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 249.9 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 185.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 146.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 460.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 138.1 MB/s  0:00:040:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 129.1 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 129.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━

In [2]:
# Prepare dataset
%cd ~/SageMaker/
!rm -rf dataset/ dataset.zip
!wget -nc -q -O "dataset.zip" "https://www.kaggle.com/api/v1/datasets/download/camnugent/california-housing-prices"

!mkdir dataset -p
!unzip -q -o ./dataset.zip -d ./dataset/

/home/ec2-user/SageMaker


In [29]:
# Import pip packages
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sagemaker.s3 import S3Uploader
from sklearn.preprocessing import LabelEncoder

In [32]:
# Prepare dataset
df = pd.read_csv('./dataset/housing.csv')
list(df)

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'median_house_value',
 'ocean_proximity']

In [33]:
print(df.shape)
print(df.isnull().sum())
df.head()

(20640, 10)
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [34]:
# Replace the null value of total_bedrooms for the mean
df['total_bedrooms'].fillna(df['total_bedrooms'].mean(), inplace=True)

/tmp/ipykernel_5353/1647323120.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['total_bedrooms'].fillna(df['total_bedrooms'].mean(), inplace=True)


In [35]:
df.drop('households', axis=1, inplace=True)
df['average_rooms']=df['total_rooms']/df['population']
df['average_bedrooms']=df['total_bedrooms']/df['population']

In [36]:
df.drop('total_rooms',axis=1,inplace=True)
df.drop('total_bedrooms',axis=1,inplace=True)

In [37]:
label_encoder = LabelEncoder()
df['ocean_proximity_encoded'] = label_encoder.fit_transform(df['ocean_proximity'])

In [38]:
print(list(df))
df.head()

['longitude', 'latitude', 'housing_median_age', 'population', 'median_income', 'median_house_value', 'ocean_proximity', 'average_rooms', 'average_bedrooms', 'ocean_proximity_encoded']


,longitude,latitude,housing_median_age,population,median_income,median_house_value,ocean_proximity,average_rooms,average_bedrooms,ocean_proximity_encoded
0,-122.23,37.88,41.0,322.0,8.3252,452600.0,NEAR BAY,2.732919,0.400621,3
1,-122.22,37.86,21.0,2401.0,8.3014,358500.0,NEAR BAY,2.956685,0.460641,3
2,-122.24,37.85,52.0,496.0,7.2574,352100.0,NEAR BAY,2.957661,0.383065,3
3,-122.25,37.85,52.0,558.0,5.6431,341300.0,NEAR BAY,2.283154,0.421147,3
4,-122.25,37.85,52.0,565.0,3.8462,342200.0,NEAR BAY,2.879646,0.495575,3


In [39]:
df.drop('ocean_proximity', axis=1, inplace=True)

In [6]:
# Set up SageMaker environment
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker"

from sagemaker import get_execution_role

role = get_execution_role()

In [7]:
# Upload dataset zip file to S3
from sagemaker.s3 import S3Uploader

inputs = S3Uploader.upload("./dataset/housing.csv", "s3://{}/{}".format(bucket, prefix))

In [41]:
from sagemaker.xgboost import XGBoost

estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="3.0-5",
    hyperparameters={
      "n_estimators": 100,
      "max_depth": 6,
      "lr": 0.05
    },
)

estimator.fit(inputs={"train": f's3://{bucket}/{prefix}/housing.csv'}, wait=True)

INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-07-02-00-18-04-032


2026-07-02 00:18:09 Starting - Starting the training job...
2026-07-02 00:18:24 Starting - Preparing the instances for training...
2026-07-02 00:18:47 Downloading - Downloading input data...
2026-07-02 00:19:32 Downloading - Downloading the training image......
2026-07-02 00:20:39 Training - Training image download completed. Training in progress.
2026-07-02 00:20:39 Uploading - Uploading generated training model/miniconda3/lib/python3.10/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-07-02:00:20:29:INFO] Imported framework sagemaker_xgboost_container.training
[2026-07-02:00:20:29:INFO] No GPUs detected (normal if no gpus installed)
[2026-07-02:00:20:29:INFO] Invoking user training script.
[2026-07-02:00:20:29:INFO] 

In [43]:
# Create new model + Update Endpoint
import boto3
import time

sm = boto3.client("sagemaker")

model = estimator.create_model(
    name = f'housing-model-{int(time.time())}'
)
model_name = model.name
model._create_sagemaker_model(instance_type="ml.m5.large")

new_config = f'xgb-endpoint-configuration-{int(time.time())}'
sm.create_endpoint_config(
    EndpointConfigName=new_config,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.large"
    }]
)

INFO:sagemaker:Creating model with name: housing-model-1782951904


{'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint-config/xgb-endpoint-configuration-1782951905',
 'ResponseMetadata': {'RequestId': '2d08ab17-3710-4580-857b-f450c544d1c2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '2d08ab17-3710-4580-857b-f450c544d1c2',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '118',
   'date': 'Thu, 02 Jul 2026 00:25:05 GMT'},
  'RetryAttempts': 0}}

In [44]:
sm.create_endpoint(
    EndpointName = "xgb-endpoint",
    EndpointConfigName = new_config
)

# sm.update_endpoint(
#     EndpointName = "xgb-endpoint",
#     EndpointConfigName = new_config
# )

{'EndpointArn': 'arn:aws:sagemaker:us-east-1:200148130345:endpoint/xgb-endpoint',
 'ResponseMetadata': {'RequestId': 'c736a524-9e21-40c1-bb2e-6fd6e597f6a7',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'c736a524-9e21-40c1-bb2e-6fd6e597f6a7',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '80',
   'date': 'Thu, 02 Jul 2026 00:25:17 GMT'},
  'RetryAttempts': 0}}

In [50]:
# Inference test
import json, boto3

client = boto3.client("sagemaker-runtime")
endpoint_name = 'xgb-endpoint'

body = {"instance": [8.3252, 41.0, 6.9841, 1.0238, 322.0, 2.5556, 37.88, -122.23]}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(body),
)
result = json.load(response["Body"])
print(result)
print(result['prediction'][0])

{'prediction': [22.7217960357666]}
22.7217960357666
